
Step 1 - Profiling Section Complete


In [1]:
import pandas as pd

In [2]:
salesdf = pd.read_csv("sales_messy.csv")
customersdf = pd.read_csv("customers.csv")

In [3]:
salesdf.shape

(208, 9)

In [4]:
salesdf.info()

<class 'pandas.DataFrame'>
RangeIndex: 208 entries, 0 to 207
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   order_id     208 non-null    int64  
 1   order_date   208 non-null    str    
 2   customer_id  200 non-null    float64
 3   country      208 non-null    str    
 4   category     208 non-null    str    
 5   product      208 non-null    str    
 6   quantity     208 non-null    int64  
 7   unit_price   195 non-null    float64
 8   discount     188 non-null    float64
dtypes: float64(3), int64(2), str(4)
memory usage: 14.8 KB


In [5]:
salesdf.isnull().sum()

order_id        0
order_date      0
customer_id     8
country         0
category        0
product         0
quantity        0
unit_price     13
discount       20
dtype: int64

In [6]:
salesdf.duplicated().sum()

np.int64(8)

In [7]:
salesdf["country"].unique()

<StringArray>
[     'GERMANY',      'Germany',       'France',      ' France',
   'Kazakhstan',           'UK', ' kazakhstan ',          'uk ',
       'Poland',          'usa',          'USA',       'Russia']
Length: 12, dtype: str

The dataset has duplicate rows, missing values in discount, unit_price and some rows without customer_id values. It also has inconsistent country names (like different capitalization and spacing, e.g. usa and USA)


Step 2 — Cleaning Section Verified


In [8]:
salesdf = salesdf.drop_duplicates()

In [9]:
salesdf["country"] = salesdf["country"].str.strip().str.title()

In [10]:
salesdf["discount"] = salesdf["discount"].fillna(0)

In [11]:
medianprice = salesdf["unit_price"].median()
salesdf["unit_price"] = salesdf["unit_price"].fillna(medianprice)

In [12]:
salesdf = salesdf.dropna(subset=["customer_id"])

In [13]:
salesdf["order_date"] = pd.to_datetime(salesdf["order_date"])

In [14]:
salesdf.isnull().sum()

order_id       0
order_date     0
customer_id    0
country        0
category       0
product        0
quantity       0
unit_price     0
discount       0
dtype: int64


Enrichment: correct revenue formula and month extracted


In [15]:
salesdf["revenue"] = salesdf["quantity"] * salesdf["unit_price"] * (1 - salesdf["discount"])

In [16]:
salesdf["month"] = salesdf["order_date"].dt.month


Step 3 — Merge Without Surprises


In [17]:
salesdf["customer_id"] = salesdf["customer_id"].astype(int)
customersdf["customer_id"] = customersdf["customer_id"].astype(int)

In [18]:
before = salesdf.shape[0]

In [19]:
salesdf = salesdf.merge(customersdf, on="customer_id", how="left")

In [20]:
after = salesdf.shape[0]
print(before)
print(after)

193
193


In [21]:
assert after == before


Step 4 — Three Aggregations


In [22]:
category_rev = (salesdf.groupby("category")["revenue"].sum().sort_values(ascending=False).reset_index())
category_rev

,category,revenue
0,Laptops,161187.4000
1,Phones,63403.4000
2,Monitors,58295.5500
3,Accessories,10323.5465


In [23]:
month_rev = (salesdf.groupby("month")["revenue"].sum().sort_values(ascending=False).reset_index())
month_rev

,month,revenue
0,7,42529.4260
1,10,33697.7500
2,8,30827.4315
3,4,26456.2405
4,6,24754.8375
5,5,23633.5065
6,12,22739.7510
7,3,19836.5880
8,2,19631.0780
9,11,19117.3905


In [24]:
segment_rev = (salesdf.groupby("segment")["revenue"].sum().sort_values(ascending=False).reset_index())
segment_rev

,segment,revenue
0,Consumer,174850.8365
1,Education,77782.0320
2,Business,40577.0280


In [25]:
print(salesdf["revenue"].sum())

print(category_rev["revenue"].sum())
print(month_rev["revenue"].sum())
print(segment_rev["revenue"].sum())

293209.89650000003
293209.8965
293209.8965
293209.8965



Step 5 — Draft Conclusions


In [26]:
total_revenue = salesdf["revenue"].sum()

top_category = category_rev.iloc[0]
top_month = month_rev.iloc[0]
top_segment = segment_rev.iloc[0]

category_share = (top_category["revenue"] / total_revenue * 100)

print("Total revenue:", total_revenue)
print("Top category revenue:", top_category["revenue"])
print("Top month revenue:", top_month["revenue"])
print("Top segment revenue:", top_segment["revenue"])
print("Category share:", category_share)

Total revenue: 293209.89650000003
Top category revenue: 161187.4
Top month revenue: 42529.426
Top segment revenue: 174850.8365
Category share: 54.97338320570635


1. The best-selling category made 161,187.4 in revenue, which is about 55% of all sales. This means most money comes from one category.

2. The best month made 42,529.4 in revenue, so sales change from month to month.

3. The top customer segment made 174,850.8 in revenue, so one group of customers buys the most.

4. Overall, sales are not evenly spread,a few categories and segments bring in most of the money.